<a href="https://colab.research.google.com/github/SpyC0der77/vscolab/blob/master/vscolab_standard_persistent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
import subprocess
import zipfile
from pathlib import Path


def _is_valid_vsix(path: Path) -> bool:
    with path.open("rb") as f:
        return f.read(2) == b"PK"


def _extension_id_from_vsix(vsix_path: Path) -> str:
    with zipfile.ZipFile(vsix_path) as zf:
        data = json.loads(zf.read("extension/package.json"))
    return f"{data['publisher']}.{data['name']}-{data['version']}"


def _ensure_vsix(vsix_path: Path, url: str, label: str) -> None:
    if vsix_path.exists() and _is_valid_vsix(vsix_path):
        return
    if vsix_path.exists():
        print(f"Removing invalid VSIX at {vsix_path}", flush=True)
        vsix_path.unlink()
    print(f"Downloading {label}...", flush=True)
    subprocess.run(["wget", "--show-progress", "-O", str(vsix_path), url], check=True)
    if not _is_valid_vsix(vsix_path):
        raise RuntimeError(
            f"Downloaded file at {vsix_path} is not a valid VSIX (expected zip archive)."
        )


def _ensure_server_settings(user_data_dir: Path) -> None:
    settings_dir = user_data_dir / "User"
    settings_dir.mkdir(parents=True, exist_ok=True)
    settings_path = settings_dir / "settings.json"
    settings = {}
    if settings_path.exists():
        settings = json.loads(settings_path.read_text())
    settings["extensions.verifySignature"] = False
    settings["security.workspace.trust.enabled"] = False
    settings["security.workspace.trust.startupPrompt"] = "never"
    settings_path.write_text(json.dumps(settings, indent=2) + "\n")


def _extensions_dir(user_data_dir: Path) -> Path:
    path = user_data_dir / "extensions"
    path.mkdir(parents=True, exist_ok=True)
    return path


def _install_vsix_manually(vsix_path: Path, user_data_dir: Path, ext_id: str) -> None:
    target = _extensions_dir(user_data_dir) / ext_id
    if target.exists():
        return
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(vsix_path) as zf:
        for member in zf.namelist():
            if not member.startswith("extension/") or member.endswith("/"):
                continue
            rel = member[len("extension/") :]
            dest = target / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            dest.write_bytes(zf.read(member))
    print(f"Extension extracted to {target}", flush=True)


def _cli_base(server_bin: Path, user_data_dir: Path) -> list[str]:
    return [
        str(server_bin),
        "--force",
        "--accept-server-license-terms",
        "--user-data-dir",
        str(user_data_dir),
        "--extensions-dir",
        str(_extensions_dir(user_data_dir)),
    ]


def _install_marketplace(server_bin: Path, ext_id: str, user_data_dir: Path) -> None:
    ext_dir = _extensions_dir(user_data_dir)
    if any(ext_dir.glob(f"{ext_id}*")):
        print(f"{ext_id} already installed", flush=True)
        return

    print(f"Installing {ext_id}...", flush=True)
    result = subprocess.run(
        [*_cli_base(server_bin, user_data_dir), "--install-extension", ext_id],
        capture_output=True,
        text=True,
    )
    if result.stdout:
        print(result.stdout, end="", flush=True)
    if result.stderr:
        print(result.stderr, end="", flush=True)
    if result.returncode != 0:
        raise RuntimeError(f"Failed to install extension {ext_id}.")


def _install_vsix(
    server_bin: Path,
    vsix_path: Path,
    user_data_dir: Path,
    ext_id: str,
) -> None:
    ext_dir = _extensions_dir(user_data_dir) / ext_id
    if ext_dir.exists():
        print(f"{ext_id} already installed at {ext_dir}", flush=True)
        return

    print(f"Installing {ext_id}...", flush=True)
    result = subprocess.run(
        [*_cli_base(server_bin, user_data_dir), "--install-extension", str(vsix_path)],
        capture_output=True,
        text=True,
    )
    if result.stdout:
        print(result.stdout, end="", flush=True)
    if result.stderr:
        print(result.stderr, end="", flush=True)

    if result.returncode == 0:
        return

    print("CLI install failed, extracting VSIX manually...", flush=True)
    _install_vsix_manually(vsix_path, user_data_dir, ext_id)
    if not (_extensions_dir(user_data_dir) / ext_id).exists():
        raise RuntimeError(f"Failed to install extension {ext_id}.")


def install_extensions(
    server_bin: Path,
    extensions: list,
    user_data_dir: Path,
    cache_dir: Path,
) -> None:
    user_data_dir = Path(user_data_dir)
    user_data_dir.mkdir(parents=True, exist_ok=True)
    _ensure_server_settings(user_data_dir)

    if not extensions:
        return

    for ext in extensions:
        if isinstance(ext, str):
            _install_marketplace(server_bin, ext, user_data_dir)
            continue

        vsix = ext["vsix"]
        vsix_path = Path(vsix)
        if not vsix_path.is_absolute():
            vsix_path = cache_dir / vsix

        label = ext.get("id") or vsix_path.name
        if ext.get("url"):
            if vsix_path.exists() and _is_valid_vsix(vsix_path):
                print(f"Using cached {label} at {vsix_path}", flush=True)
            else:
                _ensure_vsix(vsix_path, ext["url"], label)
        elif not vsix_path.exists():
            raise FileNotFoundError(f"VSIX not found: {vsix_path}")

        ext_id = ext.get("id") or _extension_id_from_vsix(vsix_path)
        _install_vsix(server_bin, vsix_path, user_data_dir, ext_id)

"""Official Microsoft VS Code web server bootstrap for vscolab.

Downloads ``server-linux-x64-web`` (Microsoft Marketplace, Copilot Chat capable)
and starts ``bin/code-server`` on a local port for Colab's proxy.
"""

from __future__ import annotations

import json
import subprocess
import time
import urllib.request
from pathlib import Path

LATEST_API = (
    "https://update.code.visualstudio.com/api/latest/server-linux-x64-web/stable"
)
SERVER_URL_TMPL = (
    "https://update.code.visualstudio.com/commit:{commit}/server-linux-x64-web/stable"
)


def resolve_commit(commit: str = "") -> tuple[str, str]:
    """Return ``(commit, version_name)``. Empty commit resolves latest stable."""
    if commit:
        return commit, commit[:12]
    with urllib.request.urlopen(LATEST_API, timeout=60) as resp:
        data = json.loads(resp.read().decode())
    return str(data["version"]), str(data.get("name") or data["version"][:12])


def _download(url: str, dest: Path, label: str) -> None:
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 0:
        print(f"Using cached {label} at {dest}", flush=True)
        return
    print(f"Downloading {label}...", flush=True)
    subprocess.run(
        ["wget", "--show-progress", "-O", str(dest), url],
        check=True,
    )


def prepare_vscode(
    cache_dir: Path,
    user_data_dir: Path,
    commit: str = "",
) -> dict:
    """Download and extract the official VS Code web server. Returns paths."""
    cache_dir = Path(cache_dir).resolve()
    user_data_dir = Path(user_data_dir).resolve()
    cache_dir.mkdir(parents=True, exist_ok=True)
    user_data_dir.mkdir(parents=True, exist_ok=True)

    resolved, version_name = resolve_commit(commit)
    print(f"VS Code {version_name} (commit {resolved[:12]})", flush=True)

    server_root = cache_dir / f"vscode-web-{resolved}"
    server_bin = server_root / "bin" / "code-server"
    tarball = cache_dir / f"vscode-server-linux-x64-web-{resolved}.tar.gz"

    if not server_bin.exists():
        _download(
            SERVER_URL_TMPL.format(commit=resolved),
            tarball,
            "VS Code web server",
        )
        if server_root.exists():
            # Incomplete extract from a prior run
            subprocess.run(["rm", "-rf", str(server_root)], check=False)
        server_root.mkdir(parents=True, exist_ok=True)
        print("Extracting VS Code web server...", flush=True)
        subprocess.run(
            [
                "tar",
                "-xzf",
                str(tarball),
                "-C",
                str(server_root),
                "--strip-components",
                "1",
            ],
            check=True,
        )

    if not server_bin.exists():
        raise RuntimeError(f"code-server not found at {server_bin}")

    extensions_dir = user_data_dir / "extensions"
    extensions_dir.mkdir(parents=True, exist_ok=True)

    return {
        "commit": resolved,
        "version": version_name,
        "server_bin": server_bin,
        "server_root": server_root,
        "user_data_dir": user_data_dir,
        "extensions_dir": extensions_dir,
    }


def start_vscode_web(
    prepared: dict,
    folder: str | Path,
    port: int = 3000,
    host: str = "0.0.0.0",
) -> None:
    """Start official ``code-server`` (web) in the background."""
    folder = str(Path(folder).resolve())
    server_bin = prepared["server_bin"]
    user_data_dir = prepared["user_data_dir"]
    extensions_dir = prepared["extensions_dir"]

    print(f"Starting VS Code (folder: {folder})...", flush=True)
    cmd = [
        str(server_bin),
        "--host",
        host,
        "--port",
        str(port),
        "--without-connection-token",
        "--accept-server-license-terms",
        "--user-data-dir",
        str(user_data_dir),
        "--extensions-dir",
        str(extensions_dir),
    ]
    # Prefer opening the workspace directly when the flag exists.
    cmd.extend(["--default-folder", folder])

    subprocess.Popen(cmd)
    time.sleep(5)
    print(f"VS Code running on port {port} — {folder}", flush=True)


def vscode_proxy_url(base_proxy_url: str, folder: str | Path) -> str:
    """Append ``?folder=`` so the workbench opens the workspace if needed."""
    folder = str(Path(folder).resolve())
    base = base_proxy_url.rstrip("/")
    sep = "&" if "?" in base else "?"
    return f"{base}{sep}folder={folder}"

"""Google Drive persistence + official VS Code serve-web bootstrap for vscolab.

Syncs the workspace with ``MyDrive/vscolab/`` on Drive.
Load: pull once (Drive -> workspace). Runtime: push-only background sync.
Pre-installs extensions from ``EXTENSIONS`` (marketplace IDs and/or VSIX).
"""

import subprocess
import threading
import time
from pathlib import Path

from google.colab import drive, output

SYNC_INTERVAL = 5
DRIVE_STORE = Path("/content/drive/MyDrive/vscolab")
IGNORE_FILE = ".vscolabignore"
DEFAULT_IGNORE = """\
# Patterns here stay on the Colab VM only — never synced to Drive.
__pycache__/
*.py[cod]
node_modules/
.venv/
venv/
*.egg-info/
.git/
"""

PORT = 3000
GIT_REPO = ""
# Pin a commit hash to freeze the VS Code build, or leave empty for latest stable.
COMMIT = ""
EXTENSIONS = [
    # Marketplace IDs (Microsoft Marketplace):
    # "ms-python.python",
    # VSIX from URL:
    # {"vsix": "name.vsix", "url": "https://..."},
]


class Persistence:
    def __init__(self, workspace: Path):
        self.workspace = workspace.resolve()

    @property
    def drive_store(self) -> Path:
        return DRIVE_STORE

    @property
    def data_dir(self) -> Path:
        return DRIVE_STORE / "data"

    @property
    def cache_dir(self) -> Path:
        return DRIVE_STORE / "cache"

    def mount(self) -> None:
        print("Mounting Google Drive...", flush=True)
        drive.mount("/content/drive")
        for d in (self.drive_store, self.data_dir, self.cache_dir):
            d.mkdir(parents=True, exist_ok=True)
        print(f"vscolab folder: {self.drive_store}", flush=True)

    def _ignore_file(self) -> Path:
        path = self.drive_store / IGNORE_FILE
        if not path.exists():
            path.write_text(DEFAULT_IGNORE)
        return path

    def _sync_ignore_for_pull(self) -> None:
        self.workspace.mkdir(parents=True, exist_ok=True)
        (self.workspace / IGNORE_FILE).write_text(self._ignore_file().read_text())

    def _sync_ignore_for_push(self) -> None:
        ws_ignore = self.workspace / IGNORE_FILE
        drive_ignore = self._ignore_file()
        if ws_ignore.exists():
            drive_ignore.write_text(ws_ignore.read_text())
        else:
            ws_ignore.write_text(drive_ignore.read_text())

    def _ensure_writable(self) -> None:
        subprocess.run(["chmod", "-R", "u+w", str(self.workspace)], check=False)

    def pull(self) -> None:
        print(f"Pulling from Drive into {self.workspace}...", flush=True)
        self._sync_ignore_for_pull()
        subprocess.run(
            [
                "rsync", "-rl",
                "--no-perms", "--no-owner", "--no-group",
                "--filter=:- .vscolabignore",
                "--exclude=data/", "--exclude=cache/",
                f"{self.drive_store}/", f"{self.workspace}/",
            ],
            check=False,
        )
        self._ensure_writable()

    def push(self) -> None:
        self._sync_ignore_for_push()
        subprocess.run(
            [
                "ionice", "-c3", "nice", "-n", "19",
                "rsync", "-rl",
                "--size-only",
                "--no-perms", "--no-owner", "--no-group",
                "--filter=:- .vscolabignore",
                "--delete", "--delete-excluded",
                "--exclude=data/", "--exclude=cache/",
                f"{self.workspace}/", f"{self.drive_store}/",
            ],
            check=False,
        )

    def start_push_loop(self) -> None:
        def loop():
            while True:
                time.sleep(SYNC_INTERVAL)
                self.push()

        threading.Thread(target=loop, daemon=True).start()
        print(
            f"Background push every {SYNC_INTERVAL}s -> Drive (see {IGNORE_FILE})",
            flush=True,
        )


folder = Path("/content/workspace")
if GIT_REPO:
    name = GIT_REPO.rstrip("/").removesuffix(".git").split("/")[-1]
    folder = Path(f"/content/{name}")

p = Persistence(workspace=folder)
p.mount()
p.pull()

if GIT_REPO and not folder.exists():
    print(f"Cloning {GIT_REPO}...", flush=True)
    subprocess.run(["git", "clone", "--progress", GIT_REPO, str(folder)], check=True)
    p.push()

prepared = prepare_vscode(p.cache_dir, p.data_dir, COMMIT)
install_extensions(prepared["server_bin"], EXTENSIONS, p.data_dir, p.cache_dir)

p.push()
p.start_push_loop()

folder = str(folder.resolve())
start_vscode_web(prepared, folder, PORT)
print(f"Drive storage: {p.drive_store}", flush=True)

# CELL 2
proxy = output.eval_js(f'google.colab.kernel.proxyPort({PORT}, {{"cache": false}})')
url = vscode_proxy_url(proxy, folder)
print(f"Open VS Code: {url}", flush=True)